In [1]:
import pandas as pd
import numpy as np

In [2]:
destination_master = pd.read_csv(
    "../data/processed/destination_master.csv"
)

destination_features = pd.read_csv(
    "../data/processed/destination_features.csv"
)

feature_matrix = pd.read_csv(
    "../data/processed/destination_feature_matrix.csv"
)

print("Destination master:", destination_master.shape)
print("Destination features:", destination_features.shape)
print("Feature matrix:", feature_matrix.shape)

Destination master: (76, 12)
Destination features: (76, 15)
Feature matrix: (76, 13)


In [3]:
def recommend_destinations(
    preferred_type=None,
    min_rating=0,
    top_n=5
):
    results = destination_features.copy()

    # Filter by destination type
    if preferred_type:
        results = results[
            results["Location_Type"].str.casefold()
            == preferred_type.casefold()
        ]

    # Filter by minimum historical rating
    results = results[
        results["Average_Rating"] >= min_rating
    ]

    # Rank using historical rating and review popularity
    results["Recommendation_Score"] = (
        0.7 * results["Rating_Score"]
        + 0.3 * results["Review_Popularity"]
    )

    results = results.sort_values(
        "Recommendation_Score",
        ascending=False
    )

    columns = [
        "Location_Name",
        "Located_City",
        "Location_Type",
        "Average_Rating",
        "Review_Count",
        "Recommendation_Score"
    ]

    return results[columns].head(top_n).reset_index(drop=True)

In [4]:
beach_recommendations = recommend_destinations(
    preferred_type="Beaches",
    min_rating=4.0,
    top_n=5
)

display(beach_recommendations)

,Location_Name,Located_City,Location_Type,Average_Rating,Review_Count,Recommendation_Score
0,Bentota Beach,Bentota,Beaches,4.59,233,0.817681
1,Passikudah Beach,Kalkudah,Beaches,4.56,160,0.753081
2,Mirissa Beach,Mirissa,Beaches,4.27,217,0.748673
3,Nilaveli Beach,Nilaveli,Beaches,4.33,165,0.716896
4,Hikkaduwa Beach,Hikkaduwa,Beaches,4.02,213,0.701671


In [5]:
def recommend_destinations(
    preferred_type=None,
    min_rating=0,
    travel_month=None,
    top_n=5
):
    results = destination_features.copy()

    # Filter by destination type
    if preferred_type:
        results = results[
            results["Location_Type"].str.casefold()
            == preferred_type.casefold()
        ]

    # Filter by minimum historical rating
    results = results[
        results["Average_Rating"] >= min_rating
    ]

    # Optional month preference
    if travel_month is not None:
        travel_month = int(travel_month)

        results = results[
            results["Most_Common_Travel_Month"] == travel_month
        ]

    # Baseline recommendation score
    results["Recommendation_Score"] = (
        0.7 * results["Rating_Score"]
        + 0.3 * results["Review_Popularity"]
    )

    results = results.sort_values(
        "Recommendation_Score",
        ascending=False
    )

    columns = [
        "Location_Name",
        "Located_City",
        "Location_Type",
        "Average_Rating",
        "Review_Count",
        "Most_Common_Travel_Month",
        "Recommendation_Score"
    ]

    return results[columns].head(top_n).reset_index(drop=True)

In [6]:
display(
    recommend_destinations(
        preferred_type="Beaches",
        min_rating=4.0,
        travel_month=7,
        top_n=5
    )
)

,Location_Name,Located_City,Location_Type,Average_Rating,Review_Count,Most_Common_Travel_Month,Recommendation_Score
0,Arugam Bay,Arugam Bay,Beaches,4.06,187,7,0.687533


In [7]:
july_beaches = recommend_destinations(
    preferred_type="Beaches",
    min_rating=4.0,
    travel_month=7,
    top_n=5
)

print("Number of recommendations:", len(july_beaches))
display(july_beaches)

Number of recommendations: 1


,Location_Name,Located_City,Location_Type,Average_Rating,Review_Count,Most_Common_Travel_Month,Recommendation_Score
0,Arugam Bay,Arugam Bay,Beaches,4.06,187,7,0.687533


In [8]:
beach_recommendations = recommend_destinations(
    preferred_type="Beaches",
    min_rating=4.0,
    travel_month=None,
    top_n=5
)

display(beach_recommendations)

,Location_Name,Located_City,Location_Type,Average_Rating,Review_Count,Most_Common_Travel_Month,Recommendation_Score
0,Bentota Beach,Bentota,Beaches,4.59,233,1,0.817681
1,Passikudah Beach,Kalkudah,Beaches,4.56,160,8,0.753081
2,Mirissa Beach,Mirissa,Beaches,4.27,217,2,0.748673
3,Nilaveli Beach,Nilaveli,Beaches,4.33,165,8,0.716896
4,Hikkaduwa Beach,Hikkaduwa,Beaches,4.02,213,1,0.701671


In [9]:
def explain_recommendation(row):
    return (
        f"{row['Location_Name']} is recommended because "
        f"it has an average tourist rating of "
        f"{row['Average_Rating']}/5 and "
        f"{int(row['Review_Count'])} tourist reviews."
    )

beach_recommendations["Recommendation_Reason"] = (
    beach_recommendations.apply(explain_recommendation, axis=1)
)

display(beach_recommendations)

,Location_Name,Located_City,Location_Type,Average_Rating,Review_Count,Most_Common_Travel_Month,Recommendation_Score,Recommendation_Reason
0,Bentota Beach,Bentota,Beaches,4.59,233,1,0.817681,Bentota Beach is recommended because it has an...
1,Passikudah Beach,Kalkudah,Beaches,4.56,160,8,0.753081,Passikudah Beach is recommended because it has...
2,Mirissa Beach,Mirissa,Beaches,4.27,217,2,0.748673,Mirissa Beach is recommended because it has an...
3,Nilaveli Beach,Nilaveli,Beaches,4.33,165,8,0.716896,Nilaveli Beach is recommended because it has a...
4,Hikkaduwa Beach,Hikkaduwa,Beaches,4.02,213,1,0.701671,Hikkaduwa Beach is recommended because it has ...


In [10]:
# Validate recommendation results

print("Number of recommendations:", len(beach_recommendations))

print(
    "Duplicate destinations:",
    beach_recommendations["Location_Name"].duplicated().sum()
)

print(
    "Missing recommendation reasons:",
    beach_recommendations["Recommendation_Reason"].isna().sum()
)

print(
    "Scores in valid range:",
    beach_recommendations["Recommendation_Score"].between(0, 1).all()
)

Number of recommendations: 5
Duplicate destinations: 0
Missing recommendation reasons: 0
Scores in valid range: True
